# Adaptive Multi-Agent GenAI Tutor — Dataset Generation Pipeline

This notebook walks through the step-by-step pipeline to transform raw lecture slide PDFs into structured, high-quality instruction-tuning datasets. 

### Fine-Tuning Philosophy
Rather than injecting factual knowledge (which will be supplied at inference time using **RAG**), our goal is to teach the model **specific cognitive behaviors and styles** corresponding to three distinct agent personas:
1. **Tutor Mode**: Structured explaining (analogy, simple terms, course grounded, quick checking).
2. **Examiner Mode**: Quiz generation and formal evaluation/feedback on student answers.
3. **Critic/Reflection Mode**: Evaluating and improving generic/weak LLM outputs.

Let's import dependencies and explore the pipeline.

In [1]:
import os
import json
import pandas as pd
from pathlib import Path

# Change directory to project root if running from notebooks folder
if Path.cwd().name == 'notebooks':
    os.chdir('..')
print(f"Project root working directory: {Path.cwd()}")

Project root working directory: /Users/salmaheshamsalem/Desktop/genai_project


## Step 1 — PDF Text Extraction

We extract text page-by-page from raw slide PDFs in `data/raw_pdfs/` using `PyMuPDF`. The extracted text is normalized to clean duplicate spaces and whitespaces while preserving structural formatting like equations and bullet points. We then write individual JSONL files and a combined dataset.

In [ ]:
# Run the text extraction script
!python3 src/extract_pdf_text.py

### Show Sample Extracted Pages
Let's load the combined page file and inspect a page metadata and extracted content.

In [ ]:
combined_pages_path = Path("data/extracted_text/all_lectures_pages.jsonl")
pages = []
with open(combined_pages_path, "r", encoding="utf-8") as f:
    for i in range(3):
        line = f.readline()
        if line:
            pages.append(json.loads(line))

for p in pages:
    print("=" * 80)
    print(f"Lecture: {p['lecture_file']} | Page: {p['page']} | Char Count: {p['char_count']}")
    print("-" * 80)
    # print first 300 chars
    print(p['text'][:400] + "...\n")

## Step 2 — Page/Slide Chunking

We chunk the slides by slide/page boundary. If a page contains very high word counts, we split it into smaller sub-chunks around 800 words. During this phase, we also automatically predict the topic for each chunk using keyword-based heuristics (defined in `src/utils.py`).

In [ ]:
# Run the chunking script
!python3 src/chunk_lectures.py

### Show Sample Chunks and Topic Classification
Let's load the chunk summary CSV and see our chunking metadata.

In [ ]:
df_summary = pd.read_csv("data/metadata/chunk_summary.csv")
print(f"Total chunks generated: {len(df_summary)}")
print("\nSample metadata rows:")
display(df_summary.head(10))

print("\nDistribution of guessed topics across slides:")
display(df_summary['topic_guess'].value_counts())

## Step 3 — Generate Synthetic Instruction Tuning Datasets

We run `src/generate_synthetic_dataset.py` to create the Tutor, Examiner, and Critic datasets. 

The generator employs a **hybrid approach**:
- If a `GROQ_API_KEY` or `OPENAI_API_KEY` is present, it uses LLM API calls with structured prompt systems to synthesize data.
- If no API keys are present, it falls back to our deterministic procedural template engine that builds beautifully formatted, high-fidelity Q&A and feedback dialogues based on our rich domain knowledge map. This guarantees the pipeline runs 100% out-of-the-box.

In [ ]:
# Run the dataset generation script
!python3 src/generate_synthetic_dataset.py

## Step 4 — Dataset Validation and Cleaning

To guarantee that our training datasets meet strict formatting schemas for Supervised Fine-Tuning, we run `src/validate_dataset.py` which:
1. Validates that Tutor outputs contain all **5 required tutoring sections**.
2. Validates Examiner outputs match the Question-Choices or Score-Feedback schema.
3. Validates Critic outputs contain both Critique and Improved Answer blocks.
4. Filters duplicates and outputs clean, standardized versions.

In [ ]:
# Run validation and display statistics
!python3 src/validate_dataset.py

## Step 5 — Displaying Representative Examples

Let's load our cleaned datasets and display 3 representative examples from each persona dataset to inspect their quality and structural styles.

In [ ]:
def show_examples(file_path, mode_name, count=3):
    print("=" * 100)
    print(f"                   3 REPRESENTATIVE EXAMPLES: {mode_name.upper()} DATASET")
    print("=" * 100)
    
    examples = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                examples.append(json.loads(line.strip()))
                if len(examples) == count:
                    break
                    
    for idx, ex in enumerate(examples, 1):
        print(f"\n[Example {idx}] Mode: {ex['mode'].upper()}")
        print(f"Instruction: {ex['instruction']}")
        print(f"Input:\n{ex['input']}")
        print(f"\nOutput:\n{ex['output']}")
        print("-" * 100)

show_examples("data/finetuning/tutor_dataset_clean.jsonl", "tutor", 3)

In [ ]:
show_examples("data/finetuning/examiner_dataset_clean.jsonl", "examiner", 3)

In [ ]:
show_examples("data/finetuning/critic_dataset_clean.jsonl", "critic", 3)